In [33]:
from google.cloud import bigquery
import pandas as pd

# Create BigQuery client
PROJECT = "students-group3"
client = bigquery.Client()


In [34]:
query_ratings = """
SELECT userId, movieId, rating
FROM `master-ai-cloud.MoviePlatform.ratings`
"""

df_ratings_sample = client.query(query_ratings).to_dataframe()
df_ratings_sample


,userId,movieId,rating
0,1,204,0.5
1,1,256,0.5
2,1,277,0.5
3,1,719,0.5
4,1,45950,0.5
...,...,...,...
105334,668,93040,5.0
105335,668,98154,5.0
105336,668,101862,5.0
105337,668,106916,5.0


In [35]:
df_ratings_sample.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   userId   105339 non-null  Int64  
 1   movieId  105339 non-null  Int64  
 2   rating   105339 non-null  float64
dtypes: Int64(2), float64(1)
memory usage: 2.6 MB


In [36]:
df_ratings_sample.describe()


,userId,movieId,rating
count,105339.0,105339.0,105339.000000
mean,364.924539,13381.312477,3.516850
std,197.486905,26170.456869,1.044872
min,1.0,1.0,0.500000
25%,192.0,1073.0,3.000000
50%,383.0,2497.0,3.500000
75%,557.0,5991.0,4.000000
max,668.0,149532.0,5.000000


In [37]:
query_movies = """
SELECT movieId, title, genres
FROM `master-ai-cloud.MoviePlatform.movies`
"""

df_movies_sample = client.query(query_movies).to_dataframe()
df_movies_sample


,movieId,title,genres
0,126929,Li'l Quinquin ( ),(no genres listed)
1,135460,Pablo (2012),(no genres listed)
2,138863,The Big Broadcast of 1936 (1935),(no genres listed)
3,141305,Round Trip to Heaven (1992),(no genres listed)
4,141472,The 50 Year Argument (2014),(no genres listed)
...,...,...,...
10324,85896,Tribute to a Bad Man (1956),Western
10325,103570,Dead Man's Burden (2012),Western
10326,105223,Colorado Territory (1949),Western
10327,128360,The Hateful Eight (2015),Western


In [38]:
print("Ratings sample shape:", df_ratings_sample.shape)
print("Movies sample shape:", df_movies_sample.shape)


Ratings sample shape: (105339, 3)
Movies sample shape: (10329, 3)


In [39]:
df_movies_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  Int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: Int64(1), object(2)
memory usage: 252.3+ KB


#  Data Cleaning


In [40]:
# Make a copy
df_ratings = df_ratings_sample.copy()

# 1. Remove missing values
df_ratings = df_ratings.dropna(subset=['userId', 'movieId', 'rating'])

# 2. Make sure types are correct
df_ratings['userId'] = df_ratings['userId'].astype(int)
df_ratings['movieId'] = df_ratings['movieId'].astype(int)
df_ratings['rating'] = df_ratings['rating'].astype(float)

# 3. Optional: Remove duplicates
df_ratings = df_ratings.drop_duplicates(subset=['userId','movieId'])

# Check result
df_ratings


,userId,movieId,rating
0,1,204,0.5
1,1,256,0.5
2,1,277,0.5
3,1,719,0.5
4,1,45950,0.5
...,...,...,...
105334,668,93040,5.0
105335,668,98154,5.0
105336,668,101862,5.0
105337,668,106916,5.0


In [41]:
# Make a copy
df_movies = df_movies_sample.copy()

# 1. Remove missing movie titles or IDs
df_movies = df_movies.dropna(subset=['movieId', 'title'])

# 2. Ensure correct types
df_movies['movieId'] = df_movies['movieId'].astype(int)
df_movies['title'] = df_movies['title'].astype(str)
df_movies['genres'] = df_movies['genres'].astype(str)

# 3. Split genres into a list
df_movies['genres_list'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies = df_movies.drop(columns=['genres'])


# Check result
df_movies


,movieId,title,genres_list
0,126929,Li'l Quinquin ( ),[(no genres listed)]
1,135460,Pablo (2012),[(no genres listed)]
2,138863,The Big Broadcast of 1936 (1935),[(no genres listed)]
3,141305,Round Trip to Heaven (1992),[(no genres listed)]
4,141472,The 50 Year Argument (2014),[(no genres listed)]
...,...,...,...
10324,85896,Tribute to a Bad Man (1956),[Western]
10325,103570,Dead Man's Burden (2012),[Western]
10326,105223,Colorado Territory (1949),[Western]
10327,128360,The Hateful Eight (2015),[Western]


## Sauvegarder

In [42]:
from pandas_gbq import to_gbq

to_gbq(
    df_movies,
    "MovieData.movies_cleaned",
    project_id="students-group3",
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 8719.97it/s]


In [43]:
from pandas_gbq import to_gbq

to_gbq(
    df_ratings,
    "MovieData.ratings_cleaned",
    project_id="students-group3",
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 7752.87it/s]


In [44]:
from google.cloud import bigquery

client = bigquery.Client(project="students-group3")

query = """
SELECT *
FROM `students-group3.MovieData.movies_cleaned`
LIMIT 5
"""

df_check = client.query(query).to_dataframe()
df_check


,movieId,title,genres_list
0,117867,'71 (2014),"[Action, Drama, Thriller, War]"
1,97757,'Hellboy': The Seeds of Creation (2004),"[Action, Adventure, Comedy, Documentary, Fantasy]"
2,26564,'Round Midnight (1986),"[Drama, Musical]"
3,779,'Til There Was You (1997),"[Drama, Romance]"
4,2072,"'burbs, The (1989)",[Comedy]


In [45]:
from google.cloud import bigquery

client = bigquery.Client(project="students-group3")

query = """
SELECT *
FROM `students-group3.MovieData.ratings_cleaned`
LIMIT 5
"""

df_check = client.query(query).to_dataframe()
df_check


,userId,movieId,rating
0,1,204,0.5
1,1,256,0.5
2,1,277,0.5
3,1,719,0.5
4,1,45950,0.5
